# Data Cleaning Pipeline

In [124]:
import pandas as pd
from fuzzywuzzy import process
import re
import numpy as np

In [133]:
# Sample DataFrame
data = {
    'User ID': ['12345', '44556', '12345', '44556', '77889'],   
    'Full Name': [' John Doe ', 'alice SMITH', 'Bob Johnson', 'Eve Adams ', 'CHRIS EVANS '],  
    'Date of Birth': ['2025-07-10', '1995-03-25', '2000-11-15', '2024-01-01', '2010-09-30'],  
    'Email': ['USER@GMAIL.COM ', 'alice.smith@yahoo.com', 'invalid-email.com', ' eve@outlook.com ', 'chris_evans@EMAIL.com'],  
    'Age': ['Twenty', '30', '45', '18', 'Unknown'],  
    'Subscription': ['monthly', 'yearly', 'MOTHLY', 'liftime', 'mnthyl'],  
    'Measurement': ['1.6bnp4hp', '2.5kw', '3.7', '1.5L', '4.3a8b'],  
    'Phone Number': ['123-456-7890', '9876543210', '(555) 123-4567', '444-5555', '999.888.7777']  
}

df= pd.DataFrame(data)

# Convert to DataFrame and transpose it
df_long = pd.DataFrame(data).T  # Transpose

# Reset index so the column names remain in the first row
df_long.reset_index(inplace=True)
print(df)
df_raw=df

  User ID     Full Name Date of Birth                  Email      Age  \
0   12345     John Doe     2025-07-10        USER@GMAIL.COM    Twenty   
1   44556   alice SMITH    1995-03-25  alice.smith@yahoo.com       30   
2   12345   Bob Johnson    2000-11-15      invalid-email.com       45   
3   44556    Eve Adams     2024-01-01       eve@outlook.com        18   
4   77889  CHRIS EVANS     2010-09-30  chris_evans@EMAIL.com  Unknown   

  Subscription Measurement    Phone Number  
0      monthly   1.6bnp4hp    123-456-7890  
1       yearly       2.5kw      9876543210  
2       MOTHLY         3.7  (555) 123-4567  
3      liftime        1.5L        444-5555  
4       mnthyl      4.3a8b    999.888.7777  


## Phase 1: Broad Cleaning (Applied to All Columns)

### Functions

#### Standardize Column Names


   - Convert to lowercase, remove spaces, replace special characters.  

In [4]:
def standardize_column_names(df):
    """Convert column names to lowercase, replace spaces with underscores."""
    df.columns = (
        df.columns.str.strip()       # Remove leading/trailing spaces
                .str.lower()         # Convert to lowercase
                .str.replace(' ', '_', regex=True) # Replace spaces with underscores
    )
    return df

#### Fix Data Types  #TODO!


   - Convert numeric-looking text columns to numbers.  
   - Convert date columns to proper datetime format.  
   - Ensure boolean values are consistent.  

#### Trim Whitespace & Standardize Case  


   - Remove leading/trailing spaces.  
   - Convert text to lowercase for consistency.  

In [5]:
def trim_whitespace(df):
    """Trim whitespace from all string columns."""
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df

#### Correcting specific values

In [138]:
feat='User ID'
df[[feat]].value_counts()

User ID
12345      2
44556      2
77889      1
Name: count, dtype: int64

In [139]:
# Step 1: Get unique values
unique_vals = df[feat].unique()
# Step 2: Create initial mapping dictionary (identity mapping)
initial_mapping = {val: val for val in unique_vals}
print("Initial Mapping:\n", initial_mapping)

Initial Mapping:
 {'12345': '12345', '44556': '44556', '77889': '77889'}


In [141]:
edited_mapping = {
    '12345': '123',
    '44556': '445',
    '77889': '77889'
}
df['New User ID'] = df[feat].map(edited_mapping)

In [142]:
print(df[['User ID', 'New User ID']])

  User ID New User ID
0   12345         123
1   44556         445
2   12345         123
3   44556         445
4   77889       77889


### Testing

In [8]:
# Before
df.head()

,User ID,Full Name,Date of Birth,Email,Age,Subscription,Measurement,Phone Number
0,12345,John Doe,2025-07-10,USER@GMAIL.COM,Twenty,monthly,1.6bnp4hp,123-456-7890
1,67890,alice SMITH,1995-03-25,alice.smith@yahoo.com,30,yearly,2.5kw,9876543210
2,11223,Bob Johnson,2000-11-15,invalid-email.com,45,MONTHLY,3.7,(555) 123-4567
3,44556,Eve Adams,2024-01-01,eve@outlook.com,18,lifetime,1.5L,444-5555
4,77889,CHRIS EVANS,2010-09-30,chris_evans@EMAIL.com,Unknown,monthyl,4.3a8b,999.888.7777


In [20]:
# Apply Phase 1
df = standardize_column_names(df)
df.head()

,user_id,full_name,date_of_birth,email,age,subscription,measurement,phone_number
0,12345,John Doe,2025-07-10,USER@GMAIL.COM,Twenty,monthly,1.6bnp4hp,123-456-7890
1,67890,alice SMITH,1995-03-25,alice.smith@yahoo.com,30,yearly,2.5kw,9876543210
2,11223,Bob Johnson,2000-11-15,invalid-email.com,45,MONTHLY,3.7,(555) 123-4567
3,44556,Eve Adams,2024-01-01,eve@outlook.com,18,liftime,1.5L,444-5555
4,77889,CHRIS EVANS,2010-09-30,chris_evans@EMAIL.com,Unknown,monthyl,4.3a8b,999.888.7777


In [ ]:
df[['user_id','date_of_birth']].iloc[0]

user_id               12345
date_of_birth    2025-07-10
Name: 0, dtype: object

In [ ]:
df = trim_whitespace(df)
df.head()

## Phase 2: Targeted Cleaning (Specific to Data Type or Context)

In [108]:
df= pd.DataFrame(data)
df.head()

,User ID,Full Name,Date of Birth,Email,Age,Subscription,Measurement,Phone Number
0,12345,John Doe,2025-07-10,USER@GMAIL.COM,Twenty,monthly,1.6bnp4hp,123-456-7890
1,67890,alice SMITH,1995-03-25,alice.smith@yahoo.com,30,yearly,2.5kw,9876543210
2,11223,Bob Johnson,2000-11-15,invalid-email.com,45,MOTHLY,3.7,(555) 123-4567
3,44556,Eve Adams,2024-01-01,eve@outlook.com,18,liftime,1.5L,444-5555
4,77889,CHRIS EVANS,2010-09-30,chris_evans@EMAIL.com,Unknown,mnthyl,4.3a8b,999.888.7777


### Functions

#### Dictionary for renaming columns

In [68]:
rename_dict = {
    'User ID': 'user_id',
    'Full Name': 'full_name',
    'Subscription': 'subscription'
}

# Apply renaming
df.rename(columns=rename_dict, inplace=True)

df.columns

Index(['user_id', 'full_name', 'Date of Birth', 'Email', 'Age', 'subscription',
       'Measurement', 'Phone Number'],
      dtype='object')

#### Clean Categorical Variables


   - Standardize known categories using a predefined dictionary.  

In [69]:
# # Dictionary for standardizing subscription values
# subscription_dict = {
#     'monthly': 'Monthly',
#     'MONTHLY': 'Monthly',
#     'yearly': 'Yearly',
#     'YEARLY': 'Yearly'
# }

# # Apply mapping to the 'subscription_plan' column
# df['subscription'] = df['subscription'].replace(subscription_dict)

# print(df['subscription'])


   - Correct spelling errors using predefined mapping and fuzzy matching for unknown errors.  

In [70]:
# Extended function with logging
def fuzzy_correct_with_log(value, valid_values, threshold=80, log_list=None):
    best_match, score = process.extractOne(value, valid_values)
    if score >= threshold:
        if log_list is not None:
            log_list.append({'original': value, 'corrected': best_match, 'score': score})
        return best_match
    else:
        if log_list is not None:
            log_list.append({'original': value, 'corrected': value, 'score': score})
        return value

In [74]:
correction_log = []
# Define valid values
valid_subscriptions = ['Monthly', 'Yearly', 'Lifetime']

# Apply correction with logging
df['subscription'] = df['subscription'].apply(
    lambda x: fuzzy_correct_with_log(x, valid_subscriptions, threshold=50, log_list=correction_log)
)

In [75]:
print("Cleaned DataFrame:")
df[['user_id','subscription']].head()

Cleaned DataFrame:


,user_id,subscription
0,12345,Monthly
1,67890,Yearly
2,11223,Monthly
3,44556,Lifetime
4,77889,Monthly


In [76]:
log_df = pd.DataFrame(correction_log)
print("\nCorrection Log:")
print(log_df)


Correction Log:
   original corrected  score
0   Monthly   Monthly    100
1    Yearly    Yearly    100
2   Monthly   Monthly    100
3  Lifetime  Lifetime    100
4    mnthyl   Monthly     77


#### Clean Numeric Variables

Extract Numeric Values from Measurement Columns  
   - Extract numeric values from mixed-format columns (e.g., `"1.5kw"` → `1.5`).  
   - Flag non-standard measurement formats containing multiple non-numeric segments (e.g., `"1.6bnp4hp"`). How to handle these and substitute a value for them.

In [79]:
# Function to extract numeric values and flag non-standard formats
def clean_measurement(value):
    # Extract all numeric parts
    numbers = re.findall(r"\d+\.\d+|\d+", value)  # Capture decimals and integers
    extracted_value = float(numbers[0]) if numbers else np.nan  # Convert to float if found
    
    # Count non-numeric segments
    non_numeric_parts = re.findall(r"[a-zA-Z]+", value)
    
    if len(non_numeric_parts) > 1:  # More than one non-numeric part = non-standard
        return "Invalid"  # Substitute an appropriate value (e.g., np.nan or 'Invalid')
    
    return extracted_value

In [80]:
features=['Measurement']
df[features]

,Measurement
0,1.6bnp4hp
1,2.5kw
2,3.7
3,1.5L
4,4.3a8b


In [92]:
# Apply function
df['cleaned_measurement'] = df['Measurement'].apply(clean_measurement)

print(df[['Measurement','cleaned_measurement']])

  Measurement cleaned_measurement
0   1.6bnp4hp             Invalid
1       2.5kw                 2.5
2         3.7                 3.7
3        1.5L                 1.5
4      4.3a8b             Invalid


In [93]:
# Filter rows with 'Invalid' cleaned values
invalid_measurements = df[df['cleaned_measurement'] == "Invalid"]['Measurement'].unique()

# Print them as dictionary keys for manual mapping
manual_map_template = {val: None for val in invalid_measurements}
print(manual_map_template)


{'1.6bnp4hp': None, '4.3a8b': None}


In [94]:
manual_map={'1.6bnp4hp': 1.6, '4.3a8b': 4.3}

In [ ]:
df['cleaned_measurement'] = df.apply(
    lambda row: manual_map[row['Measurement']] if row['cleaned_measurement'] == 'Invalid' and row['Measurement'] in manual_map
    else row['cleaned_measurement'],axis=1)

print(df[['Measurement','cleaned_measurement']])

  Measurement  cleaned_measurement
0   1.6bnp4hp                  1.6
1       2.5kw                  2.5
2         3.7                  3.7
3        1.5L                  1.5
4      4.3a8b                  4.3


#### Apply Rules  

   - Example: Assign an "age_group" based on an age threshold.  dd

In [120]:
df= pd.DataFrame(data)
print(df[['Age']])

       Age
0   Twenty
1       30
2       45
3       18
4  Unknown


In [121]:
# Define custom mapping for known non-numeric values
age_replacements = {
    'Twenty': 20,
    'Unknown': 9999
}

# Apply replacements
df['Age'] = df['Age'].replace(age_replacements)

# Convert to numeric (optional safety if other strings exist)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age'] = df['Age'].astype(int)  # or .astype('Int64') to allow NA

In [122]:
print(df[['Age']])

    Age
0    20
1    30
2    45
3    18
4  9999


In [125]:
# Define age groups using np.select
conditions = [
    df['Age'] < 20,
    df['Age'].between(20, 35, inclusive='left'),
    df['Age'].between(35, 100, inclusive='left')
]
choices = ['Young Adult', 'Adult', 'Old']

df['age_group'] = np.select(conditions, choices, default='Unknown')

In [126]:
print(df[['Age', 'age_group']])

    Age    age_group
0    20        Adult
1    30        Adult
2    45          Old
3    18  Young Adult
4  9999      Unknown


## Phase 3: Final Checks & Quality Assurance

8. Re-check Missing Values  
   - Identify remaining gaps and decide on appropriate handling (e.g., imputation, removal).  


9. Summary Statistics  
   - Generate descriptive stats to validate the dataset's integrity.  
